# DQN Training Notebook

**Deep Q-Network (DQN)** is a reinforcement learning algorithm that trains a neural network to approximate the Q-function — the expected future reward for taking each action from a given state. Instead of learning from one step at a time (which is noisy and correlated), DQN stores past experiences in a *replay buffer* and learns from random batches. A separate *target network* is periodically synced from the main network, which stabilizes training by preventing the Q-targets from shifting every step.

This notebook trains a DQN agent to play the BWSI SGAI game: the agent observes the current ambulance state and the attributes of the humanoid at the door, then decides whether to save, skip, squish, or scram. It mirrors `training/train_rl.py` exactly, with argparse replaced by hardcoded constants you can edit directly.

In [1]:
%matplotlib inline

In [2]:
import os
import sys
from pathlib import Path

# Ensure the project root is the working directory and is on sys.path,
# so local packages (game_interfaces, gameplay, models) are importable
# and relative paths like DATA_DIR="data" resolve correctly.
# Works whether Jupyter was launched from bwsi-sgai/ or training/.
_here = Path().resolve()
_root = _here.parent if _here.name == "training" else _here
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import os
import random
import matplotlib.pyplot as plt
import numpy as np
import torch

from game_interfaces.data_parser import DataParser
from game_interfaces.train_env import TrainEnv
from gameplay.constants import AMBULANCE_CAPACITY, DATA_DIR, RL_MODEL_PATH, SHIFT_LENGTH
from gameplay.scorekeeper import ScoreKeeper
from models.CNNClassifier import CNNClassifier
from models.DQNAgent import DQNAgent

## Hyperparameters

These are the same constants defined at the top of `train_rl.py`. Changing them here changes training behavior — this is where your experiments start.

Key things to try:
- **`NUM_EPISODES`** — more episodes = more experience, but slower to run. 500 is a weak baseline; 2000+ produces more consistent behavior.
- **`EPSILON_DECAY`** — controls how quickly the agent shifts from random exploration to learned policy. If you set this too low (fast decay), the agent commits to bad habits early. Too high (slow decay), it never exploits what it learned.
- **`GAMMA`** — the discount factor. Values close to 1.0 make the agent care about long-term reward; values close to 0 make it greedy for immediate reward.
- **`LEARNING_RATE`** — the Adam optimizer step size. Too large → unstable training. Too small → very slow to learn.

In [4]:
# ── Hyperparameters — edit these to experiment ──────────────────────────────
NUM_EPISODES: int = 2500
BATCH_SIZE: int = 32          # transitions per gradient step
BUFFER_CAPACITY: int = 5000   # max replay buffer size
EPSILON_DECAY: int = 1250     # episodes over which epsilon decays to its minimum
TARGET_UPDATE_FREQ: int = 20  # episodes between target net hard updates
GAMMA: float = 0.99           # discount factor
LEARNING_RATE: float = 1e-3   # Adam step size

# Log a summary line every N episodes
LOG_INTERVAL: int = 50

# Seed for reproducibility (set to None to disable)
SEED: int | None = 42

# Where to load data from, where to save weights, and where to write plots
TRAINING_DATA_DIR: str = DATA_DIR
SAVE_PATH: str = RL_MODEL_PATH
RL_LOG_DIR: str = "training/RL-logs"

print(f"Will train for {NUM_EPISODES} episodes.")
print(f"Weights will be saved to: {SAVE_PATH}")
print(f"Plots will be saved to: {RL_LOG_DIR}")

Will train for 2500 episodes.
Weights will be saved to: models\weights\baselineRL.pth
Plots will be saved to: training/RL-logs


## Seeding

Setting a fixed seed makes your experiments *reproducible* — running the notebook twice with the same seed produces the same training trajectory. This matters when you're comparing two hyperparameter settings: if you don't seed, you can't tell whether a difference in results came from the hyperparameters or from random luck.

We seed Python's `random`, NumPy, and PyTorch (CPU and GPU) so every random draw during training — including experience sampling from the buffer, network weight initialization, and epsilon-greedy action selection — is deterministic.

In [5]:
def set_seed(seed: int) -> None:
    """Seed all random number generators for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


if SEED is not None:
    set_seed(SEED)
    print(f"Seeded with {SEED}.")
else:
    print("Seeding disabled — results will vary between runs.")

Seeded with 42.


## Environment + Data

`TrainEnv` is a headless version of the game environment — it runs the game logic without opening any windows, so it can step through thousands of episodes quickly. It wraps the game's data parser (which reads humanoid records from the CSV) and a scorekeeper (which tracks score and enforces ambulance capacity).

We set `display=False` so no Tkinter window opens. The observation returned by `env.reset()` and `env.step()` is a flat 1D numpy array describing the current state.

In [6]:
def make_env(data_dir: str, seed: int | None = None) -> TrainEnv:
    """Build a fresh TrainEnv from the data directory."""
    data_parser = DataParser(data_dir=data_dir, seed=seed)
    scorekeeper = ScoreKeeper(shift_length=SHIFT_LENGTH, capacity=AMBULANCE_CAPACITY, seed=seed)
    classifier = CNNClassifier(img_data_root=data_dir)
    return TrainEnv(
        root=None,
        w=0,
        h=0,
        data_parser=data_parser,
        scorekeeper=scorekeeper,
        classifier=classifier,
        display=False,
    )


env = make_env(TRAINING_DATA_DIR, seed=SEED)
print(f"Observation size: {env.obs_size}")
print(f"Number of actions: {env.action_space.n}")

Observation size: 15
Number of actions: 4


## Training Loop

Here is the core DQN algorithm. Each episode is one full game playthrough:

1. **Reset** — get the initial observation from a fresh game state.
2. **Step** — at each turn, the agent picks an action using **epsilon-greedy** exploration: with probability ε it picks randomly (exploration), otherwise it picks the action with the highest predicted Q-value (exploitation). ε starts high (lots of exploration) and decays toward 0 over `EPSILON_DECAY` episodes.
3. **Push to replay buffer** — store the `(obs, action, reward, next_obs, done)` transition. The buffer holds the last `BUFFER_CAPACITY` transitions; older ones are discarded.
4. **Train step** — sample a random minibatch from the buffer, compute the DQN loss (Bellman residual), and do one gradient step. Training only starts once the buffer has at least `BATCH_SIZE` transitions.
5. **End-of-episode** — decay epsilon; every `TARGET_UPDATE_FREQ` episodes, hard-copy the policy net weights to the target net.

**Save logic note:** We save the weights from the single highest-reward episode seen so far. Watch the ε value on save lines — if the agent saves a "best" model at ε > 0.5, that episode was mostly random luck, not learned behavior.

> **Running from the CLI instead of this notebook?** Python buffers stdout when not connected to a terminal, so the per-episode log lines may not appear until training finishes. Use `python -u main.py train --type rl` to force unbuffered output and see progress in real time.

In [7]:
agent = DQNAgent(
    obs_size=env.obs_size,
    num_actions=env.action_space.n,
    lr=LEARNING_RATE,
    gamma=GAMMA,
    epsilon_decay=EPSILON_DECAY,
    buffer_capacity=BUFFER_CAPACITY,
    batch_size=BATCH_SIZE,
    target_update_freq=TARGET_UPDATE_FREQ,
)

best_reward = float("-inf")
weights_saved = False
episode_rewards = []
epsilon_log = []

for episode in range(1, NUM_EPISODES + 1):
    obs, _ = env.reset()
    episode_reward = 0.0
    episode_loss = 0.0
    steps = 0
    done = False

    while not done:
        # eps-greedy: random action with probability eps, greedy otherwise
        action = agent.select_action(obs)
        next_obs, reward, done, _truncated, _info = env.step(action)

        # Push raw obs -- DQNAgent.train_step normalizes internally.
        # Use done or _truncated: truncated episodes are also terminal for
        # bootstrap purposes — without this, Q-targets add γ·V(next_obs)
        # on a state the agent will never actually continue from.
        agent.replay_buffer.push(obs, action, reward, next_obs, done or _truncated)

        loss = agent.train_step()
        if loss is not None:
            episode_loss += loss

        obs = next_obs
        episode_reward += reward
        steps += 1

    # End-of-episode bookkeeping: epsilon decay + (maybe) target net update
    target_updated = agent.end_episode()
    episode_rewards.append(episode_reward)
    epsilon_log.append(agent.epsilon)

    # Save the weights from the single highest-reward episode seen so far.
    # Watch out: early episodes run with high epsilon (lots of random actions),
    # so a lucky random episode can set best_reward before the agent has learned
    # anything useful. The epsilon value printed on save tells you whether this
    # happened — a save at ε > 0.5 means the "best" model is mostly random.
    epsilon = agent.epsilon
    if episode_reward > best_reward and epsilon < 0.5:
        best_reward = episode_reward
        os.makedirs(os.path.dirname(SAVE_PATH) or ".", exist_ok=True)
        agent.save(SAVE_PATH)
        weights_saved = True
        print(
            f"  -> new best {episode_reward:+.1f} at ep {episode} "
            f"(eps={epsilon:.2f}) -- saved to {SAVE_PATH}"
        )

    if episode % LOG_INTERVAL == 0:
        recent = episode_rewards[-LOG_INTERVAL:]
        avg = sum(recent) / len(recent)
        avg_loss = episode_loss / steps if steps > 0 else 0.0
        target_flag = " [target updated]" if target_updated else ""
        print(
            f"Episode {episode:4d}/{NUM_EPISODES}  "
            f"reward: {episode_reward:+.1f}  "
            f"avg(last {LOG_INTERVAL}): {avg:+.1f}  "
            f"loss(ep): {avg_loss:.4f}  "
            f"epsilon: {agent.epsilon:.3f}  "
            f"steps: {steps}"
            f"{target_flag}"
        )

if weights_saved:
    print(f"\nTraining complete. Best episode reward: {best_reward:+.1f}")
    print(f"Weights saved to: {SAVE_PATH}")
else:
    print(
        "\nTraining complete. No weights saved — epsilon never dropped below 0.5."
    )
    print("Try increasing NUM_EPISODES or decreasing EPSILON_DECAY.")

Episode   50/2500  reward: +0.5  avg(last 50): -0.4  loss(ep): 0.0670  epsilon: 0.962  steps: 16
Episode  100/2500  reward: +0.8  avg(last 50): -0.9  loss(ep): 0.0510  epsilon: 0.924  steps: 16 [target updated]
Episode  150/2500  reward: +2.4  avg(last 50): -0.3  loss(ep): 0.0751  epsilon: 0.886  steps: 19
Episode  200/2500  reward: -1.2  avg(last 50): +0.1  loss(ep): 0.0636  epsilon: 0.848  steps: 16 [target updated]
Episode  250/2500  reward: +1.2  avg(last 50): +0.0  loss(ep): 0.0939  epsilon: 0.810  steps: 13
Episode  300/2500  reward: -3.0  avg(last 50): +0.2  loss(ep): 0.0801  epsilon: 0.772  steps: 21 [target updated]
Episode  350/2500  reward: -2.5  avg(last 50): -0.2  loss(ep): 0.1329  epsilon: 0.734  steps: 18
Episode  400/2500  reward: +0.4  avg(last 50): +0.3  loss(ep): 0.1490  epsilon: 0.696  steps: 16 [target updated]
Episode  450/2500  reward: -2.5  avg(last 50): +1.0  loss(ep): 0.2526  epsilon: 0.658  steps: 21
Episode  500/2500  reward: +0.9  avg(last 50): +1.1  loss(e

## Results

A summary of what was achieved across all training episodes. Use this to judge whether training converged and whether the saved weights are actually learned behavior (not lucky randomness).

In [8]:
# Rolling average over the final window of episodes.
# A flat or rising trend means the agent learned something useful.
window = min(50, len(episode_rewards))
final_avg = sum(episode_rewards[-window:]) / window
print(f"Final epsilon       : {agent.epsilon:.4f}")
print(f"Avg reward (last {window} episodes): {final_avg:+.1f}")
print(f"Best episode reward : {best_reward:+.1f}")

Final epsilon       : 0.0500
Avg reward (last 50 episodes): +7.1
Best episode reward : +13.7


## Training Plots

`training_curves.png` is saved to `RL_LOG_DIR`. It has two subplots:

- **Top** — episode reward (faint) and 50-episode moving average (solid). A rising trend in the moving average means the agent is learning. A flat or oscillating average means it may have converged or is stuck.
- **Bottom** — epsilon decay. Shows when the agent transitions from mostly-random exploration to mostly-greedy exploitation. A best-reward save at high epsilon means the saved weights reflect luck more than learning.

In [9]:
def _save_rl_plots(
    episode_rewards: list[float],
    epsilon_log: list[float],
    log_dir: str,
) -> None:
    """Save RL training curve and epsilon decay plots to log_dir."""
    os.makedirs(log_dir, exist_ok=True)
    episodes_range = range(1, len(episode_rewards) + 1)
    window = min(50, len(episode_rewards))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

    # ── Top: episode reward + moving average ──────────────────────────────
    ax1.plot(episodes_range, episode_rewards, alpha=0.4, color="steelblue", label="Episode reward")
    if len(episode_rewards) >= window:
        moving_avg = np.convolve(episode_rewards, np.ones(window) / window, mode="valid")
        ma_range = range(window, len(episode_rewards) + 1)
        ax1.plot(ma_range, moving_avg, color="steelblue", linewidth=2, label=f"{window}-ep moving avg")
    ax1.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax1.set_xlabel("Episode")
    ax1.set_ylabel("Total Reward")
    ax1.set_title("Episode Reward")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # ── Bottom: epsilon decay ─────────────────────────────────────────────
    ax2.plot(episodes_range, epsilon_log, color="darkorange", linewidth=1.5)
    ax2.set_xlabel("Episode")
    ax2.set_ylabel("Epsilon")
    ax2.set_title("Exploration Rate (Epsilon Decay)")
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)

    fig.suptitle("DQN Training Curves", fontsize=14)
    plt.tight_layout()
    curves_path = os.path.join(log_dir, "training_curves.png")
    fig.savefig(curves_path, dpi=150)
    plt.close(fig)
    print(f"Training curves saved to {curves_path}")

In [10]:
_save_rl_plots(episode_rewards, epsilon_log, RL_LOG_DIR)

Training curves saved to training/RL-logs\training_curves.png
